# CRC Pi Slab — inverted-T ribs with ballast fill and top plate

Demonstrates `scite.beam.floor.CRCPiSlab` (concept *icc_apps/concepts/crc_pi_slab_bimaterial_section.md*, M2):
a one-way CRC floor whose **continuous plate is at the bottom** and whose **ribs point upward**.
Optionally a **ballast / insulation fill** (foam concrete or unbound granulate) fills the space
between — and possibly above — the ribs, and a **concrete top plate** closes the section.

```
      |<---------------- b_eff ---------------->|
      +-----------------------------------------+   top plate (opt.)
      |       fill above the webs (opt.)        |
      +----------------+-------+----------------+   y = H_rib
      |      fill      |  web  |      fill      |
      +----------------+-------+----------------+   y = H_bay
      |   o grid           o bars               |   plate (tension flange)
      +-----------------------------------------+   y = 0
```

All components share **one linear strain profile** (rigid connection) and are integrated with the
single kernel `CrossSection.integrate`; each carries its own constitutive law. The fill has zero
tensile stiffness and strength and **softens** after its peak (`FoamConcrete`, Sargin law).

*Material data of the fill are indicative (concept §3.8) — replace them with product data.*

In [ ]:
import contextlib, io, logging
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.floor import (BALLAST_PRESETS, BallastSpec, CRCPiSlab, CRCRibbedSlab,
                              LoadModel, TopPlateSpec)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
logging.getLogger('matplotlib.axes._base').setLevel(logging.ERROR)   # equal-aspect sketch notes

def quiet():
    # silence the M-κ solver's progress prints
    return contextlib.redirect_stdout(io.StringIO())

# Reference geometry [mm]: H_rib = TOTAL depth of the inverted T, H_bay = plate thickness
GEO = dict(H_rib=250, H_bay=50, B_rib=90, L_rib=6000, L_bay=800)
lm = LoadModel(q_k=3.0, delta_g_k=1.0)
print('presets:', ', '.join(BALLAST_PRESETS))

---
## 1  Inverted T versus T rib

Same web, plate and CFRP bars ($A_f = 1.5$ cm², grid switched off with `r_long=0`).
The T rib carries the plate in **compression**; the Pi rib has its plate in **tension**,
so the compression zone is the narrow web: higher cracking moment (large tension flange),
lower ultimate moment.

In [ ]:
pi0 = CRCPiSlab(**GEO, r_long=0.0)
t0 = CRCRibbedSlab(**GEO)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 3.6), width_ratios=[1, 1, 1.6])
t0.rib_beam.sls.cs.plot_cross_section(ax=ax1, title='T rib (CRCRibbedSlab)', show_layer_labels=False)
pi0.rib_beam.sls.cs.plot_cross_section(ax=ax2, title='inverted T (CRCPiSlab)', show_layer_labels=False)
with quiet():
    for m, c, lbl in ((t0, '#1f77b4', 'T rib'), (pi0, '#d62728', 'inverted T')):
        mk = m.rib_beam.sls.bda.mk
        mk.plot_mk(ax=ax3, color=c, label=lbl, show_cracking_marker=True)
ax3.set_title('M-κ (SLS)')
plt.tight_layout()
for m, name in ((t0, 'T rib'), (pi0, 'inverted T')):
    mk = m.rib_beam.sls.bda.mk          # solved above
    print(f'{name:11s}  M_cr = {mk.M_cr_pos:6.2f} kNm   M_R = {np.max(mk.M_values):6.2f} kNm')

---
## 2  Ballast fill and top plate — composite section

Four variants of the Pi rib (CFRP bars + longitudinal plate grid, `r_long = 0.5`):

| variant | fill | top plate |
|---|---|---|
| A | — | — |
| B | FC600 flush with the webs (`h_fill = h_w`) | — |
| C | FC600, 60 mm above the webs | — |
| D | FC600, 60 mm above the webs | 40 mm C25 |

The sketches show the regions (fill ochre, hatched; top plate grey); the M-κ plots mark where
each component reaches its limit strain (`show_component_markers=True`) — the sweep continues past
secondary limits, the dashed line is the post-peak path.

In [ ]:
h_w = GEO['H_rib'] - GEO['H_bay']
fc600 = lambda h_fill: BallastSpec.from_preset('FC600', h_fill=h_fill)
variants = {
    'A  plain':                CRCPiSlab(**GEO),
    'B  fill flush':           CRCPiSlab(**GEO, ballast=fc600(h_w)),
    'C  fill +60':             CRCPiSlab(**GEO, ballast=fc600(h_w + 60)),
    'D  fill +60, plate 40':   CRCPiSlab(**GEO, ballast=fc600(h_w + 60), top_plate=TopPlateSpec(h_top=40)),
}
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
with quiet():
    for (name, m), ax_cs, ax_mk in zip(variants.items(), axes[0], axes[1]):
        m.rib_beam.sls.cs.plot_cross_section(ax=ax_cs, title=name, show_layer_labels=False)
        m.rib_beam.sls.bda.mk.plot_mk(ax=ax_mk, label='SLS', show_component_markers=True)
        ax_mk.set_title(name, fontsize=9)
        ax_mk.legend(fontsize=6, loc='lower right')
plt.tight_layout()

### Component limits

`MKappaAnalysis.get_component_limits()` lists, per component, where its limit strain is reached
on the raw solver path (first reached = governing).

In [ ]:
for name, m in variants.items():
    mk = m.rib_beam.sls.bda.mk          # solved above
    reached = [f"{d['label']} ({d['kind'][:4]}.) κ={d['kappa_lim']*1e3:.4f} 1/m"
               for d in mk.get_component_limits() if d['reached']]
    print(f"{name:24s} M_R = {np.max(mk.M_values):6.2f} kNm   ->  " + ('; '.join(reached) or '—'))

---
## 3  A soft fill reaches its limit first

The owner's question: can the fill be the primary failure reason? A fill with a short branch
(`eps_cu = -2 ‰`, below its peak strain) reaching 60 mm above the webs loses its capacity at
κ ≈ 0.02 1/m — marked on the curve — while the section carries on until the concrete of the
inverted T crushes. The stress profile at the M-peak shows the fill's stress block above the
concrete web.

In [ ]:
soft = CRCPiSlab(**GEO, ballast=BallastSpec(h_fill=h_w + 60, f_c=2.0, E=1250.0,
                                             eps_cu=-0.002, name='soft fill'))
with quiet():
    mk = soft.rib_beam.sls.bda.mk
    fig = plt.figure(figsize=(14, 4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.3, 1, 2])
    mk.plot_mk(ax=fig.add_subplot(gs[0]), show_component_markers=True)
    i_pk = int(np.argmax(mk.M_values))
    mk.plot_state_at_kappa(mk.kappa_values[i_pk],
                           ax_strain=fig.add_subplot(gs[1]), ax_stress=fig.add_subplot(gs[2]))
plt.tight_layout()
for d in mk.get_component_limits():
    print(f"{d['label']:14s} {d['kind']:11s} lim={d['eps_lim']*1e3:6.2f} ‰  reached={d['reached']!s:5s} "
          f"η at peak={d['eta_peak']:.2f}")

---
## 4  Floor-system assessment, GWP and cost

Self-weight includes the fill and the top plate. The fill acts structurally only in the **rib**
direction; for the transverse bay strip (the plate between the webs) it is mass only. A
**mass-only** granulate (`glass-foam gravel`, `structural=False`) adds weight but no capacity.

In [ ]:
variants['E  gravel flush (mass only)'] = CRCPiSlab(
    **GEO, ballast=BallastSpec.from_preset('glass-foam gravel', h_fill=h_w))
rows = []
with quiet():
    for name, m in variants.items():
        a = m.assess(lm)
        v = m.volumes()
        rib = a['CRC Π rib']
        rows.append((name, m.H_total, m.g_k, rib['p_R_sls'], rib['eta_ULS'],
                     v['gwp_per_m2'], v['gwp_agg'] / v['A_bay'], v['cost_per_m2']))
print(f"{'variant':28s} {'H [mm]':>6s} {'g_k':>6s} {'p_R,SLS':>8s} {'η_ULS':>6s} "
      f"{'GWP/m²':>7s} {'(fill)':>7s} {'EUR/m²':>7s}")
for r in rows:
    print(f"{r[0]:28s} {r[1]:6.0f} {r[2]:6.2f} {r[3]:8.2f} {r[4]:6.2f} {r[5]:7.1f} {r[6]:7.1f} {r[7]:7.1f}")

In [ ]:
m = variants['D  fill +60, plate 40']
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
with quiet():
    m.plot_floor_assessment(axes, load_model=lm)
fig.suptitle('Variant D — load composition, bay strip and Π rib (p–w)', fontweight='bold')
plt.tight_layout()
with quiet():
    m.rib_beam.sls.bda.F_R, m.rib_beam.uls.bda.F_R
m.report(lm)

---
### Notes

* The fill data (`BALLAST_PRESETS`) are **indicative** orders of magnitude
  (foamed-concrete correlations `E ≈ 0.4·f_c^1.2` GPa, ACI 523) — use product data.
* Foam concrete is cement-rich: its GWP per m² can exceed that of the structural concrete
  (column *(fill)* above); a mass-only granulate is cheaper in carbon but adds no capacity.
* `CrossSection.quadrature='legacy'` reproduces the pre-kernel integration rule
  (≈ 1 % higher M_R) for comparison with earlier results.